In [ ]:
import os
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
@dataclass
class DatasetStats:
  mean: np.ndarray
  std: np.ndarray

In [ ]:
class RotorWindowDataset(Dataset):
  def __init__(
      self,
      csv_path,
      window_len=120,
      step=20,
      channels=None,
      stats: DatasetStats | None = None,
      run_ids_filter=None,
      pre_fault_keep_prob=0.35,
      min_fault_fraction=0.10,
      fault_threshold=0.99,
    ):
    if channels is None:
      channels = [
                "accelerometer_x", "accelerometer_y", "accelerometer_z",
                "gyroscope_x", "gyroscope_y", "gyroscope_z",
                "angular_velocity_x", "angular_velocity_y", "angular_velocity_z",
                "angular_acceleration_x", "angular_acceleration_y", "angular_acceleration_z",
                "motor0", "motor1", "motor2", "motor3",
                ]
      self.channels = channels
      self.rotor_cols = ["rotor0_eff", "rotor1_eff", "rotor2_eff", "rotor3_eff"]
      self.window_len = window_len
      self.step = step
      self.fault_threshold = fault_threshold

      df = pd.read_csv(csv_path)

      required = ["run_id", "time"] + channels + self.rotor_cols
      missing = [c for c in required if c not in df.columns]
      if missing:
        raise ValueError(f"CSV missing columns: {missing}")

      if run_ids_filter is not None:
        df = df[df["run_id"].isin(run_ids_filter)].copy()

      df = df.sort_values(["run_id", "time"]).reset_index(drop=True)

      windows = []
      eff_labels = []
      fault_labels = []
      run_ids = []

      for run_id, group in df.groupby("run_id"):
        group = group.sort_values("time").reset_index(drop=True)
        n = len(group)

        if n < window_len:
          continue

        i = 0
        while i + window_len <= n:
          window = group.iloc[i:i + window_len]

          x = window[channels].values.astype(np.float32)
          rotor_vals = window[self.rotor_cols].values.astype(np.float32)

          # Use a small lookback average for smoother targets
          y_eff = rotor_vals[-5:].mean(axis=0).astype(np.float32)
          y_fault = (y_eff < fault_threshold).astype(np.float32)

          fault_mask = (rotor_vals < fault_threshold).any(axis=1)
          fault_fraction = fault_mask.mean()

          # Keep all clearly faulted windows.
          # Downsample purely healthy/pre-fault windows instead of hard balancing later.
          keep = True
          if fault_fraction == 0.0:
            keep = (np.random.rand() < pre_fault_keep_prob)
          elif fault_fraction < min_fault_fraction:
            keep = False

          if keep:
            windows.append(x)
            eff_labels.append(y_eff)
            fault_labels.append(y_fault)
            run_ids.append(run_id)

          i += step

      if not windows:
          raise ValueError("No windows were created. Check csv_path, channels, and window settings.")

      self.windows = np.stack(windows).astype(np.float32)
      self.eff_labels = np.stack(eff_labels).astype(np.float32)
      self.fault_labels = np.stack(fault_labels).astype(np.float32)
      self.run_ids = np.array(run_ids)

      if stats is None:
        mean = self.windows.mean(axis=(0, 1))
        std = self.windows.std(axis=(0, 1)) + 1e-6
        self.stats = DatasetStats(mean=mean, std=std)
      else:
        self.stats = stats

      self.windows = (self.windows - self.stats.mean[None, None, :]) / self.stats.std[None, None, :]


  def __len__(self):
    return len(self.windows)


  def __getitem__(self, idx):
    x = torch.tensor(self.windows[idx], dtype=torch.float32)
    y_eff = torch.tensor(self.eff_labels[idx], dtype=torch.float32)
    y_fault = torch.tensor(self.fault_labels[idx], dtype=torch.float32)
    return x, y_eff, y_fault

In [ ]:
class RotorRegressor(nn.Module):
  def __init__(self, n_channels, hidden_size=128, dropout=0.25):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv1d(n_channels, 64, kernel_size=7, padding=3),
        nn.BatchNorm1d(64),
        nn.GELU(),
        nn.Conv1d(64, 64, kernel_size=5, padding=2),
        nn.BatchNorm1d(64),
        nn.GELU(),
        nn.MaxPool1d(2),

        nn.Conv1d(64, 128, kernel_size=5, padding=2),
        nn.BatchNorm1d(128),
        nn.GELU(),
        nn.Conv1d(128, 128, kernel_size=3, padding=1),
        nn.BatchNorm1d(128),
        nn.GELU(),
        nn.MaxPool1d(2),

        nn.Conv1d(128, 192, kernel_size=3, padding=1),
        nn.BatchNorm1d(192),
        nn.GELU()
        )

    self.lstm = nn.LSTM(
        input_size=192,
        hidden_size=hidden_size,
        num_layers=2,
        dropout=dropout,
        batch_first=True,
        bidirectional=True
        )

    out_dim = hidden_size * 2

    self.shared = nn.Sequential(
        nn.Linear(out_dim, 128),
        nn.GELU(),
        nn.Dropout(dropout)
        )

    self.eff_head = nn.Sequential(
        nn.Linear(128, 64),
        nn.GELU(),
        nn.Linear(64, 4),
        nn.Sigmoid()   # keeps effectiveness in [0, 1]
        )

    self.fault_head = nn.Sequential(
        nn.Linear(128, 64),
        nn.GELU(),
        nn.Linear(64, 4)  # per-rotor fault logits
    )


  def forward(self, x):
    # x: [B, T, C]
    x = x.transpose(1, 2)          # [B, C, T]
    x = self.conv(x)               # [B, C', T']
    x = x.transpose(1, 2)          # [B, T', C']

    lstm_out, _ = self.lstm(x)
    features = lstm_out[:, -1, :]  # final timestep

    shared = self.shared(features)
    eff = self.eff_head(shared)
    fault_logits = self.fault_head(shared)
    return eff, fault_logits

In [ ]:
def make_weighted_sampler(dataset: RotorWindowDataset):
  # More weight to faulted windows
  faulty = dataset.fault_labels.any(axis=1)
  weights = np.where(faulty, 2.5, 1.0).astype(np.float32)
  return WeightedRandomSampler(
      weights=torch.tensor(weights, dtype=torch.float32),
      num_samples=len(weights),
      replacement=True
      )

In [ ]:
def evaluate(model, loader, eff_loss_fn, fault_loss_fn):
  model.eval()

  total_eff_loss = 0.0
  total_fault_loss = 0.0
  total = 0

  eff_true_all, eff_pred_all = [], []
  fault_true_all, fault_pred_all = [], []

  with torch.no_grad():
    for x, y_eff, y_fault in loader:
      x = x.to(DEVICE)
      y_eff = y_eff.to(DEVICE)
      y_fault = y_fault.to(DEVICE)

      pred_eff, pred_fault_logits = model(x)

      eff_loss = eff_loss_fn(pred_eff, y_eff)
      fault_loss = fault_loss_fn(pred_fault_logits, y_fault)

      bs = x.size(0)
      total_eff_loss += eff_loss.item() * bs
      total_fault_loss += fault_loss.item() * bs
      total += bs

      eff_true_all.append(y_eff.cpu().numpy())
      eff_pred_all.append(pred_eff.cpu().numpy())

      fault_true_all.append(y_fault.cpu().numpy())
      fault_pred_all.append((torch.sigmoid(pred_fault_logits) > 0.5).float().cpu().numpy())

  eff_true = np.vstack(eff_true_all)
  eff_pred = np.vstack(eff_pred_all)
  fault_true = np.vstack(fault_true_all)
  fault_pred = np.vstack(fault_pred_all)

  mse = np.mean((eff_true - eff_pred) ** 2)
  mae = np.mean(np.abs(eff_true - eff_pred))
  per_rotor_mae = np.mean(np.abs(eff_true - eff_pred), axis=0)

  fault_acc = (fault_true == fault_pred).mean()
  sample_fault_acc = (fault_true == fault_pred).all(axis=1).mean()

  return {
      "eff_loss": total_eff_loss / total,
      "fault_loss": total_fault_loss / total,
      "mse": mse,
      "mae": mae,
      "per_rotor_mae": per_rotor_mae,
      "fault_acc": fault_acc,
      "sample_fault_acc": sample_fault_acc
      }

In [ ]:
csv_path = "imu_readings.csv"
WINDOW_LEN = 120
STEP = 20
BATCH_SIZE = 128

all_df = pd.read_csv(csv_path)
all_run_ids = np.sort(all_df["run_id"].unique())

train_runs, temp_runs = train_test_split(all_run_ids, test_size=0.30, random_state=SEED)
val_runs, test_runs = train_test_split(temp_runs, test_size=0.50, random_state=SEED)

In [ ]:
base_train_ds = RotorWindowDataset(
    csv_path,
    window_len=WINDOW_LEN,
    step=STEP,
    run_ids_filter=train_runs,
    stats=None,
)

stats = base_train_ds.stats

train_ds = RotorWindowDataset(
    csv_path,
    window_len=WINDOW_LEN,
    step=STEP,
    run_ids_filter=train_runs,
    stats=stats,
)

val_ds = RotorWindowDataset(
    csv_path,
    window_len=WINDOW_LEN,
    step=STEP,
    run_ids_filter=val_runs,
    stats=stats,
    pre_fault_keep_prob=1.0,
    min_fault_fraction=0.0,
)

test_ds = RotorWindowDataset(
    csv_path,
    window_len=WINDOW_LEN,
    step=STEP,
    run_ids_filter=test_runs,
    stats=stats,
    pre_fault_keep_prob=1.0,
    min_fault_fraction=0.0,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=make_weighted_sampler(train_ds),
    drop_last=False,
)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 150245 | Val: 36337 | Test: 37400


In [ ]:
model = RotorRegressor(n_channels=len(train_ds.channels)).to(DEVICE)

eff_loss_fn = nn.SmoothL1Loss()
fault_loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=4, factor=0.5
    )

In [ ]:
best_val = float("inf")
patience = 8
epochs_no_improve = 0
EPOCHS = 60

for epoch in range(EPOCHS):
  model.train()
  running_loss = 0.0
  total = 0

  for x, y_eff, y_fault in train_loader:
    x = x.to(DEVICE)
    y_eff = y_eff.to(DEVICE)
    y_fault = y_fault.to(DEVICE)

    optimizer.zero_grad()

    pred_eff, pred_fault_logits = model(x)

    eff_loss = eff_loss_fn(pred_eff, y_eff)
    fault_loss = fault_loss_fn(pred_fault_logits, y_fault)

    loss = eff_loss + 0.4 * fault_loss
    loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    bs = x.size(0)
    running_loss += loss.item() * bs
    total += bs

  val_metrics = evaluate(model, val_loader, eff_loss_fn, fault_loss_fn)
  scheduler.step(val_metrics["mse"])

  print(
      f"Epoch {epoch+1:02d} | "
      f"train_loss={running_loss/total:.4f} | "
      f"val_mse={val_metrics['mse']:.4f} | "
      f"val_mae={val_metrics['mae']:.4f} | "
      f"fault_acc={val_metrics['fault_acc']:.4f} | "
      f"sample_fault_acc={val_metrics['sample_fault_acc']:.4f}"
      )

  if val_metrics["mse"] < best_val:
    best_val = val_metrics["mse"]
    epochs_no_improve = 0
    torch.save(
          {
              "model_state_dict": model.state_dict(),
              "stats_mean": train_ds.stats.mean,
              "stats_std": train_ds.stats.std,
              "channels": train_ds.channels,
              "window_len": WINDOW_LEN,
          },
            "rotor_regressor_multitask.pt",
        )
  else:
    epochs_no_improve += 1
    if epochs_no_improve >= patience:
      print("Early stopping")
      break

Epoch 01 | train_loss=0.2625 | val_mse=0.0763 | val_mae=0.2228 | fault_acc=0.7787 | sample_fault_acc=0.3554
Epoch 02 | train_loss=0.2598 | val_mse=0.0804 | val_mae=0.2248 | fault_acc=0.7650 | sample_fault_acc=0.3492
Epoch 03 | train_loss=0.2574 | val_mse=0.0783 | val_mae=0.2221 | fault_acc=0.7744 | sample_fault_acc=0.3720
Epoch 04 | train_loss=0.2544 | val_mse=0.0773 | val_mae=0.2251 | fault_acc=0.7857 | sample_fault_acc=0.3907
Epoch 05 | train_loss=0.2530 | val_mse=0.0785 | val_mae=0.2231 | fault_acc=0.7457 | sample_fault_acc=0.3232
Epoch 06 | train_loss=0.2505 | val_mse=0.0866 | val_mae=0.2262 | fault_acc=0.7119 | sample_fault_acc=0.2657
Epoch 07 | train_loss=0.2458 | val_mse=0.0792 | val_mae=0.2237 | fault_acc=0.7618 | sample_fault_acc=0.3479
Epoch 08 | train_loss=0.2436 | val_mse=0.0791 | val_mae=0.2224 | fault_acc=0.7645 | sample_fault_acc=0.3475
Epoch 09 | train_loss=0.2421 | val_mse=0.0818 | val_mae=0.2230 | fault_acc=0.7397 | sample_fault_acc=0.3127
Early stopping


In [ ]:
saved_model = torch.load("rotor_regressor_multitask.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(saved_model["model_state_dict"])

test_metrics = evaluate(model, test_loader, eff_loss_fn, fault_loss_fn)
print("\nTest metrics")
print("MSE:", test_metrics["mse"])
print("MAE:", test_metrics["mae"])
print("Per-rotor MAE:", test_metrics["per_rotor_mae"])
print("Fault acc:", test_metrics["fault_acc"])
print("Sample fault acc:", test_metrics["sample_fault_acc"])


Test metrics
MSE: 0.07697017
MAE: 0.22626567
Per-rotor MAE: [0.26716852 0.20827983 0.2107569  0.21885428]
Fault acc: 0.7665307486631016
Sample fault acc: 0.30868983957219254
